# ML-Filtered Backtest

This notebook compares the baseline residual entry rule against an ML probability filter and a probability-sized version of the same signal. The comparison uses the same event universe so that differences come from filtering and sizing rather than from different candidate events.


The baseline strategy accepts every candidate event. The ML-filtered strategy accepts only events where the predicted reversion probability exceeds the selected threshold. The probability-sized strategy accepts the same screened events and scales exposure by the predicted probability.

When real trade-level PnL is unavailable, this notebook uses a residual z-score PnL proxy. The resulting outputs are diagnostic research artifacts, not live trading results.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.backtest import run_ml_backtest_comparison, trade_count_comparison, pnl_comparison
from src.plotting import (
    plot_strategy_equity_curve,
    plot_strategy_drawdown,
    plot_strategy_bar_comparison,
)
from src.database import (
    connect_database,
    initialize_database,
    store_ml_backtest_trade_log,
    store_ml_backtest_equity_curve,
    store_ml_backtest_summary,
    store_avoided_bad_trades_analysis,
)


In [ ]:
processed_dir = ROOT / 'data' / 'processed'
figures_dir = ROOT / 'figures'
database_path = ROOT / 'data' / 'database' / 'triangular_stat_arb.sqlite'
schema_path = ROOT / 'sql' / 'schema.sql'

labels_path = processed_dir / 'event_labels_table.csv'
predictions_path = processed_dir / 'predicted_reversion_probabilities.csv'

labels = pd.read_csv(labels_path)
predictions = pd.read_csv(predictions_path)

overlap = set(labels['event_id']).intersection(set(predictions['event_id']))
if not overlap:
    rng = np.random.default_rng(13)
    placeholder = labels.loc[:, ['event_id', 'triplet_id', 'method', 'event_date', 'label']].copy()
    base = 0.38 + 0.32 * placeholder['label'].astype(float).to_numpy() + rng.normal(0.0, 0.16, size=len(placeholder))
    placeholder['predicted_reversion_probability'] = np.clip(base, 0.05, 0.95)
    placeholder['split'] = 'placeholder'
    placeholder['classification_threshold'] = 0.60
    placeholder['predicted_label'] = (placeholder['predicted_reversion_probability'] > 0.60).astype(int)
    predictions = placeholder
    predictions.to_csv(processed_dir / 'ml_backtest_placeholder_predictions.csv', index=False)

labels.head()


In [ ]:
probability_threshold = 0.60
transaction_cost = 0.05

result = run_ml_backtest_comparison(
    labels=labels,
    predictions=predictions,
    probability_threshold=probability_threshold,
    transaction_cost=transaction_cost,
)

trade_log = result['strategy_trade_log']
equity_curve = result['strategy_equity_curve']
summary = result['strategy_summary']
avoided = result['avoided_bad_trades']

summary


In [ ]:
trade_count = trade_count_comparison(summary)
pnl = pnl_comparison(summary)

trade_log.to_csv(processed_dir / 'ml_backtest_trade_log.csv', index=False)
equity_curve.to_csv(processed_dir / 'ml_backtest_equity_curve.csv', index=False)
summary.to_csv(processed_dir / 'ml_backtest_strategy_summary.csv', index=False)
trade_count.to_csv(processed_dir / 'baseline_vs_ml_trade_count.csv', index=False)
pnl.to_csv(processed_dir / 'baseline_vs_ml_pnl.csv', index=False)
avoided.to_csv(processed_dir / 'avoided_bad_trades_analysis.csv', index=False)

trade_count


In [ ]:
plot_strategy_equity_curve(equity_curve, figures_dir / 'ml_filtered_vs_baseline_equity.png')
plot_strategy_drawdown(equity_curve, figures_dir / 'ml_drawdown_comparison.png')
plot_strategy_bar_comparison(summary, 'sharpe', figures_dir / 'ml_sharpe_comparison.png', title='Sharpe comparison')
plot_strategy_bar_comparison(summary, 'turnover', figures_dir / 'ml_turnover_comparison.png', title='Turnover comparison')


In [ ]:
initialize_database(database_path, schema_path)
with connect_database(database_path) as conn:
    store_ml_backtest_trade_log(conn, trade_log)
    store_ml_backtest_equity_curve(conn, equity_curve)
    store_ml_backtest_summary(conn, summary)
    store_avoided_bad_trades_analysis(conn, avoided)

pnl


The key comparison is not whether the ML strategy produces more trades. A useful filter may trade less often if it avoids weak or failed candidate events. The next step should inspect whether accepted trades have better realized quality after transaction costs and whether the improvement survives walk-forward evaluation.
